# 03 — NLP Pipeline

Runs the sentiment analysis pipeline on scraped articles and produces NLP features.

**Uses:** `distilbert-base-uncased-finetuned-sst-2-english` via Hugging Face Transformers.

In [ ]:
import sys
sys.path.insert(0, '../models/nlp_analysis')

import json
import pandas as pd
from pathlib import Path
from transformers import pipeline as hf_pipeline
from text_preprocessor import clean_text, chunk_text

## 1. Load Sample Articles

In [ ]:
news_dir = Path('../data/raw/news_articles')
all_files = list(news_dir.glob('*.json'))
print(f'Article files found: {len(all_files)}')

# Preview one file
if all_files:
    with open(all_files[0]) as f:
        sample = json.load(f)
    print(f'Player: {sample[0]["player"]} | Articles: {len(sample)}')
    print('Sample text excerpt:')
    print(sample[0]['text'][:300])

## 2. Load Sentiment Model

In [ ]:
MODEL_NAME = 'distilbert-base-uncased-finetuned-sst-2-english'
pipe = hf_pipeline('sentiment-analysis', model=MODEL_NAME, truncation=True, max_length=512)
print('Model loaded.')

## 3. Test on Single Article

In [ ]:
if all_files:
    test_text = sample[0]['text']
    chunks = chunk_text(clean_text(test_text))
    for c in chunks[:3]:
        result = pipe(c[:512])
        print(f'{result[0]["label"]:10s} ({result[0]["score"]:.3f}) | {c[:80]}...')

## 4. Run Full Pipeline

Or just call the script:

In [ ]:
# !python ../models/nlp_analysis/sentiment_analyzer.py

## 5. Inspect NLP Features

In [ ]:
nlp_features = pd.read_csv('../data/processed/nlp_features.csv')
print(f'Players with NLP features: {len(nlp_features)}')
nlp_features.describe()

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
nlp_features['media_sentiment_score'].hist(bins=20, ax=axes[0], color='steelblue')
axes[0].set_title('Sentiment Score Distribution')
nlp_features['hype_level'].hist(bins=20, ax=axes[1], color='orange')
axes[1].set_title('Hype Level Distribution')
nlp_features['injury_concerns_score'].hist(bins=20, ax=axes[2], color='red')
axes[2].set_title('Injury Concern Score')
plt.tight_layout()